# EXP-002 — Notebook làm việc

Hồ sơ chốt: `experiments/01_baselines/EXP-002_LR/experiment.ipynb`. Mặc định chỉ đọc kết quả. Bật RUN_TRAIN khi cần chạy lại; output được lưu riêng. Nhật ký duy nhất là `experiments/experiments_logs.xlsx`.

In [1]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/wesad/models.py").is_file())
os.chdir(ROOT)
RUN_TRAIN = False
FOLDER = ROOT / 'experiments/01_baselines/EXP-002_LR'
def read_csv(path, *args, **kwargs):
    path = Path(path)
    if not path.exists() and path.name == "metrics_by_subject.csv":
        path = path.with_name("results.csv")
    return pd.read_csv(path, *args, **kwargs)

# EXP-002 — Logistic Regression baseline

Người thực hiện: Thái

Ngày 15/11/2026

## Mục tiêu
Đánh giá LR mặc định và so sánh với Dummy trên cùng bảng physiological68.

## Giả thuyết trước khi chạy
LR sử dụng thông tin đặc trưng nên có thể nhận diện các trạng thái tốt hơn Dummy.
Mức cải thiện có thể khác nhau giữa các người.

## Thiết lập
- Hai bài toán: multiclass và binary.
- R01 kiểm tra S2; R02 chạy LOSO đủ 15 người.
- Imputer median và StandardScaler được fit trong train của từng fold.
- Chưa tuning, chưa cân bằng lớp.
- Độ đo: Macro-F1 và Balanced Accuracy.


In [2]:
if RUN_TRAIN:
    from pathlib import Path
    from copy import deepcopy
    import os
    import sys
    import json
    import hashlib
    import warnings

    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "src/wesad/models.py").is_file()), None)
    if ROOT is None:
        raise RuntimeError("Không tìm thấy thư mục dự án.")
    os.chdir(ROOT)
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))

    import pandas as pd
    os.environ.setdefault("MPLCONFIGDIR", str(Path(os.environ["TEMP"]) / "wesad_matplotlib"))
    import matplotlib.pyplot as plt
    from IPython.display import display, Image
    from wesad.data import SUBJECTS, load_feature_table
    from wesad.evaluation import run_experiment
    from wesad.models import build_pipeline
    from wesad.utils import read_config, file_hash, fingerprint, environment

    plt.switch_backend("Agg")
    print("Project:", ROOT)
    print("Python:", sys.executable)


## Kiểm tra dữ liệu

In [3]:
if RUN_TRAIN:
    defaults = read_config(ROOT / "configs/defaults.yaml")
    FEATURES = ROOT / defaults["features"]
    SCHEMA = ROOT / defaults["schema"]
    assert file_hash(FEATURES) == defaults["feature_sha256"]
    df, feature_columns = load_feature_table(FEATURES, SCHEMA, SUBJECTS)
    assert len(df) == 6189 and len(feature_columns) == 68
    print("Số cửa sổ:", len(df))
    print("Số đặc trưng:", len(feature_columns))
    print("Số người:", df.subject_id.nunique())
    print("Giá trị thiếu:", int(df[feature_columns].isna().sum().sum()))
    display(pd.crosstab(df.subject_id, df.label_original).reindex(SUBJECTS))


## R01 — Kiểm tra trên S2

Train 14 người còn lại, test S2. Kiểm tra pipeline và đầu ra trước khi chạy đầy đủ.

In [4]:
if RUN_TRAIN:
    CONFIG = deepcopy(defaults)
    CONFIG.update({
        "protocol": "notebook_research_new_runs_v1",
        "experiment_id": "EXP-002",
        "group_experiment_id": "EXP-002",
        "run_no": "r01",
        "run_scope": "quick",
        "models": ["lr"],
        "tasks": ["multiclass", "binary"],
        "balance": False,
        "nested": False,
        "output_dir": "outputs/experiments/EXP-002/clean_v1/r01_quick",
        "source_paths": ["src/wesad/data.py", "src/wesad/evaluation.py",
                         "src/wesad/models.py", "src/wesad/utils.py"],
    })
    display(build_pipeline("lr", CONFIG))
    RUN_DIR = ROOT / CONFIG["output_dir"]
    result = run_experiment(CONFIG, resume=True, outer_subjects=["S2"])
    metrics = read_csv(RUN_DIR / "metrics_by_subject.csv")
    assert len(metrics) == 2 and set(metrics.subject_id) == {"S2"}
    print("Trạng thái:", result["status"])
    display(metrics)


## Kiểm tra cảnh báo R01

In [5]:
if RUN_TRAIN:
    def read_warnings(folder):
        rows = []
        for path in sorted((folder / "folds").glob("*/*/*/details.json")):
            detail = json.loads(path.read_text(encoding="utf-8"))
            assert set(detail["outer_train_subjects"]).isdisjoint(detail["outer_test_subjects"])
            for message in detail["warnings"]:
                rows.append({"task": path.parents[2].name,
                             "subject_id": path.parent.name, "warning": message})
        return pd.DataFrame(rows, columns=["task", "subject_id", "warning"])

    warnings_r01 = read_warnings(RUN_DIR)
    display(warnings_r01)
    print("Số cảnh báo R01:", len(warnings_r01))


## Nhận xét R01

- multiclass: Macro-F1 = 0.466127; Balanced Accuracy = 0.621005.
- binary: Macro-F1 = 0.414815; Balanced Accuracy = 0.496454.

Không có cảnh báo trong hai fold R01. Giữ nguyên tham số để chạy đủ 15 người.

## R02 — LR baseline đủ 15 người

Giữ nguyên thiết lập R01. Mỗi fold train trên 14 người và test người còn lại.
Không điều chỉnh tham số theo kết quả S2. Tổng cộng 30 fold cho hai bài toán.


In [6]:
if RUN_TRAIN:
    CONFIG_R02 = deepcopy(CONFIG)
    CONFIG_R02.update({"run_no": "r02", "run_scope": "main",
                       "output_dir": "outputs/experiments/EXP-002/clean_v1/r02_main"})
    RUN_DIR_R02 = ROOT / CONFIG_R02["output_dir"]
    result_r02 = run_experiment(CONFIG_R02, resume=True, outer_subjects=None)
    assert result_r02["status"] == "complete"
    metrics_r02 = read_csv(RUN_DIR_R02 / "metrics_by_subject.csv")
    assert len(metrics_r02) == 30
    assert not metrics_r02.duplicated(["task", "model", "subject_id"]).any()
    for task in ["binary", "multiclass"]:
        assert set(metrics_r02.loc[metrics_r02.task.eq(task), "subject_id"]) == set(SUBJECTS)
    print("PASS: đủ 30 fold, không trùng kết quả.")
    display(metrics_r02)
    summary_r02 = metrics_r02.groupby("task")[["macro_f1", "balanced_accuracy"]].agg(["mean", "std"])
    display(summary_r02)
    warnings_r02 = read_warnings(RUN_DIR_R02)
    display(warnings_r02)
    print("Số cảnh báo R02:", len(warnings_r02))


## So sánh với Dummy

So sánh theo cùng người test và cùng bài toán.

In [7]:
if RUN_TRAIN:
    DUMMY_DIR = ROOT / "experiments/01_baselines/EXP-001_Dummy/runs/20261114"
    dummy_manifest = json.loads((DUMMY_DIR / "config.json").read_text(encoding="utf-8"))
    lr_manifest = json.loads((RUN_DIR_R02 / "config.json").read_text(encoding="utf-8"))
    assert dummy_manifest["data_sha256"] == lr_manifest["data_sha256"]
    assert dummy_manifest["code_sha256"] == lr_manifest["code_sha256"]
    assert dummy_manifest["environment"] == lr_manifest["environment"]
    dummy_metrics = read_csv(DUMMY_DIR / "metrics_by_subject.csv")
    comparison = metrics_r02.merge(dummy_metrics, on=["task", "subject_id"],
                                  suffixes=("_lr", "_dummy"), validate="one_to_one")
    assert len(comparison) == 30
    assert comparison.n_test_lr.eq(comparison.n_test_dummy).all()
    for metric in ["macro_f1", "balanced_accuracy"]:
        comparison[metric + "_delta"] = comparison[metric + "_lr"] - comparison[metric + "_dummy"]
    display(comparison)
    display(comparison.groupby("task")[["macro_f1_delta", "balanced_accuracy_delta"]].agg(["mean", "std"]))
    for task, part in comparison.groupby("task"):
        print(task, "| tăng:", int((part.macro_f1_delta > 1e-12).sum()),
              "| giảm:", int((part.macro_f1_delta < -1e-12).sum()),
              "| gần như bằng:", int((part.macro_f1_delta.abs() <= 1e-12).sum()))
    comparison.to_csv(RUN_DIR_R02 / "comparison_by_subject.csv", index=False)


## Biểu đồ kết quả


In [8]:
if RUN_TRAIN:
    from pathlib import Path
    import os
    os.environ.setdefault("MPLCONFIGDIR", str(Path(os.environ["TEMP"]) / "wesad_matplotlib"))
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display, Image
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "src/wesad/models.py").is_file()), None)
    if ROOT is None:
        raise RuntimeError("Hãy mở notebook trong thư mục dự án.")

    RUN_DIR_VIS = ROOT / "outputs/experiments/EXP-002/clean_v1/r02_main"
    REFERENCE_VIS = ROOT / "experiments/01_baselines/EXP-001_Dummy/runs/20261114"
    MODEL_LABEL = "LR"
    REFERENCE_LABEL = "Dummy"
    FIGURES_VIS = RUN_DIR_VIS / "figures"

    for name in ["metrics_by_subject.csv", "predictions.csv"]:
        if not (RUN_DIR_VIS / name).is_file():
            raise FileNotFoundError(f"Thiếu kết quả: {RUN_DIR_VIS / name}")

    metrics_vis = read_csv(RUN_DIR_VIS / "metrics_by_subject.csv")
    predictions_vis = read_csv(RUN_DIR_VIS / "predictions.csv")
    assert not metrics_vis.duplicated(["task", "model", "subject_id"]).any()
    assert not predictions_vis.duplicated(["task", "model", "subject_id", "window_id"]).any()
    assert metrics_vis.model.nunique() == 1
    reference_vis = None
    if REFERENCE_VIS is not None:
        reference_vis = read_csv(REFERENCE_VIS / "metrics_by_subject.csv")
        paired_vis = metrics_vis.merge(reference_vis, on=["task", "subject_id"],
                                       suffixes=("_current", "_reference"), validate="one_to_one")
        assert len(paired_vis) == len(metrics_vis)
        assert paired_vis.n_test_current.eq(paired_vis.n_test_reference).all()

    FIGURES_VIS.mkdir(parents=True, exist_ok=True)
    plt.switch_backend("Agg")
    def show_figure(fig, filename):
        fig.tight_layout()
        target = FIGURES_VIS / filename
        fig.savefig(target, dpi=150, bbox_inches="tight")
        plt.close(fig)
        display(Image(filename=str(target)))

    summary_vis = metrics_vis.groupby("task")[["macro_f1", "balanced_accuracy"]].agg(["mean", "std"])
    display(summary_vis)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, metric in zip(axes, ["macro_f1", "balanced_accuracy"]):
        means = summary_vis[(metric, "mean")]
        deviations = summary_vis[(metric, "std")].fillna(0)
        ax.bar(means.index, means, yerr=deviations, capsize=5, color=["#2c7fb8", "#41ab5d"][:len(means)])
        ax.set(title=metric.replace("_", " ").title(), ylim=(0, 1.05), ylabel="Điểm")
        ax.grid(axis="y", alpha=0.2)
        for x, mean in enumerate(means):
            ax.text(x, mean + 0.02, f"{mean:.3f}", ha="center")
    fig.suptitle(f"EXP-002 — {MODEL_LABEL}")
    show_figure(fig, "summary_metrics.png")

    for task in ["binary", "multiclass"]:
        part = metrics_vis.loc[metrics_vis.task.eq(task)].copy()
        if part.empty:
            continue
        subjects = sorted(part.subject_id.unique(), key=lambda s: int(s[1:]))
        part = part.set_index("subject_id").reindex(subjects)
        fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
        for ax, metric in zip(axes, ["macro_f1", "balanced_accuracy"]):
            scores = pd.DataFrame({MODEL_LABEL: part[metric]})
            if reference_vis is not None:
                ref = reference_vis.loc[reference_vis.task.eq(task)].set_index("subject_id").reindex(subjects)
                scores[REFERENCE_LABEL] = ref[metric]
            scores.plot.bar(ax=ax, ylim=(0, 1.05), rot=0)
            ax.set_ylabel(metric.replace("_", " ").title())
            ax.grid(axis="y", alpha=0.2)
        axes[0].set_title(f"EXP-002 — {task}: kết quả theo người test")
        axes[1].set_xlabel("Người test")
        show_figure(fig, f"{task}_subject_metrics.png")

        pred = predictions_vis.loc[predictions_vis.task.eq(task)]
        assert len(pred) == int(part.n_test.sum())
        labels = [0, 1] if task == "binary" else [1, 2, 3]
        names = ["Non-stress", "Stress"] if task == "binary" else ["Baseline", "Stress", "Amusement"]
        cm = confusion_matrix(pred.y_true, pred.y_pred, labels=labels)
        row_sums = cm.sum(axis=1, keepdims=True)
        normalized = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        ConfusionMatrixDisplay(cm, display_labels=names).plot(ax=axes[0], cmap="Blues", colorbar=False, values_format="d")
        ConfusionMatrixDisplay(normalized, display_labels=names).plot(ax=axes[1], cmap="Blues", colorbar=False, values_format=".2f")
        axes[0].set_title("Số cửa sổ")
        axes[1].set_title("Tỷ lệ theo nhãn thật")
        fig.suptitle(f"EXP-002 — {task}: confusion matrix")
        show_figure(fig, f"{task}_confusion_matrix.png")

    if reference_vis is not None:
        fig, axes = plt.subplots(2, 1, figsize=(11, 6))
        for ax, task in zip(axes, ["binary", "multiclass"]):
            part = paired_vis.loc[paired_vis.task.eq(task)].copy()
            part["subject_number"] = part.subject_id.str[1:].astype(int)
            part = part.sort_values("subject_number")
            delta = part.macro_f1_current - part.macro_f1_reference
            ax.bar(part.subject_id, delta, color=["#2c7fb8" if d >= 0 else "#e6550d" for d in delta])
            ax.axhline(0, color="black", linewidth=0.8)
            ax.set(title=task, ylabel="Chênh lệch Macro-F1")
            ax.grid(axis="y", alpha=0.2)
        fig.suptitle(f"{MODEL_LABEL} so với {REFERENCE_LABEL}")
        show_figure(fig, "comparison_delta_macro_f1.png")

    print("Ảnh được lưu tại:", FIGURES_VIS)
    if metrics_vis.subject_id.nunique() == 1:
        print("Lượt kiểm tra một người; chưa dùng để kết luận cho đủ 15 người.")
    else:
        print("Cột sai số ở bảng tổng hợp là độ lệch chuẩn giữa các người.")
    print("Confusion matrix gộp cửa sổ; bảng tổng hợp tính trung bình theo người.")


In [9]:
if RUN_TRAIN:
    for task, part in metrics_r02.groupby("task"):
        print(task)
        for metric in ["macro_f1", "balanced_accuracy"]:
            print(f"{metric}: {part[metric].mean():.6f} ± {part[metric].std():.6f}")
        best = part.loc[part.macro_f1.idxmax()]
        worst = part.loc[part.macro_f1.idxmin()]
        print(f"Macro-F1 cao nhất: {best.subject_id} ({best.macro_f1:.6f})")
        print(f"Macro-F1 thấp nhất: {worst.subject_id} ({worst.macro_f1:.6f})")


## Kết quả và nhận xét R02

Đã hoàn thành LOSO 15 người cho hai bài toán, tổng cộng 30 fold.

- binary: Macro-F1 = 0.810269 ± 0.193757; Balanced Accuracy = 0.830362 ± 0.175180.
  Macro-F1 cao nhất ở S4 (1.000000), thấp nhất ở S14 (0.409222).
  So với Dummy: Macro-F1 trung bình thay đổi +0.397988; tăng ở 13 người, giảm ở 1 người.
- multiclass: Macro-F1 = 0.589391 ± 0.148416; Balanced Accuracy = 0.631471 ± 0.139367.
  Macro-F1 cao nhất ở S5 (0.796091), thấp nhất ở S10 (0.290313).
  So với Dummy: Macro-F1 trung bình thay đổi +0.354344; tăng ở 15 người, giảm ở 0 người.

Ghi nhận 0 cảnh báo trong R02.
Điểm trung bình và độ lệch chuẩn được tính theo người, mỗi người có trọng số bằng nhau.
Confusion matrix gộp các cửa sổ dự đoán ngoài mẫu nên có trọng số khác bảng trung bình theo người.

LR có thể cải thiện điểm trung bình nhưng chưa ổn định trên mọi người. Không chọn tham số dựa trên điểm outer test.

## Giới hạn

Chưa tuning, chưa cân bằng lớp và chưa kiểm định ý nghĩa thống kê. Kết quả trên WESAD chưa chứng minh hiệu quả ngoài thực tế.

## Quyết định tiếp theo

Thực hiện EXP-011 với class_weight=balanced, giữ nguyên dữ liệu, preprocessing, cách chia fold và tham số LR còn lại để kiểm tra tác động của cân bằng lớp.

In [10]:
saved = pd.read_csv(FOLDER / "results.csv")
assert len(saved) == 30
display(saved.groupby("task")[["macro_f1", "balanced_accuracy"]].agg(["mean", "std"]))

            macro_f1           balanced_accuracy          
                mean       std              mean       std
task                                                      
binary      0.810269  0.193757          0.830362  0.175180
multiclass  0.589391  0.148416          0.631471  0.139367
